# Prep for website
Output the UAV polygons for target classes
* Generate from mask in small UAV initially
* Consider filtering the UAV masks to 10m initially

In [1]:
import geopandas
import leafmap
import pandas
import pathlib
import dask.distributed
import datetime
import numpy

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Value to edit

In [2]:
website_tab = "uav_polygons"

In [10]:
all_uav_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua",
                       "Purau", "Ihutai", "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Ihutai", "LeftBank_Nov25", "ThePoint_Nov25"]

# Cells to run
* Loop over prediction sites and years
* Split out seagrass extents for each date
* Save the identifying satellite information for each date

In [4]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:60167/status,
Dashboard: http://127.0.0.1:60167/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:60176,Workers: 0
Dashboard: http://127.0.0.1:60167/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:60220,Total threads: 2
Dashboard: http://127.0.0.1:60222/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:60180,


In [8]:
data_path = utils.get_data_path()
utils.create_data_folders()
uav_folder = data_path / "classified_uav"
training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"

In [14]:
training_labels = pandas.read_csv(
        training_labels_file, sep="\t", header=None, names=["Value", "Key"]
    ).set_index('Key')['Value'].to_dict()

for uav_site in all_uav_sites:
    output_folder = data_path / "website" / "uav_polygons" / uav_site
    output_folder.mkdir(exist_ok=True, parents=True)

    uav_file = uav_folder / f"{uav_site}_classified.tif"
    uav_data = utils.load_classification(filename=uav_file, chunks=True)
    uav_data.load()

    uav_classes_present = numpy.unique(uav_data.data)

    targets = ["Seagrass", "Ulva", "Gracilaria"]

    for target in targets:
        output_file = output_folder / f"{target.lower()}_uav_polygon.gpkg"
        if output_file.exists():
            print(f"{target} UAV polygon at {uav_site} already run. Go to next")
            continue
        target_id = training_labels[target]

        if target_id not in uav_classes_present:
            geometry = geopandas.GeoDataFrame(geometry=[], crs=crs)
        
        mask = uav_data == target_id
        break
        geometry = utils.mask_to_polygons(mask).dissolve().geometry
        geometry = geopandas.GeoDataFrame(geometry, crs="2193")
        geometry.to_file(output_file)
    break


In [34]:
mask = uav_data == target_id

In [35]:
import time
start = time.perf_counter()
geometry = utils.mask_to_polygons(mask).dissolve().geometry
elapsed = time.perf_counter() - start
print(f"Elapsed time: {elapsed:.4f} seconds tp fit the model")


KeyboardInterrupt



In [ ]:
geometry